# Limpieza de datos financieros: Sesgo por repetición (deduplicación)

## 1. Concepto
El **sesgo por repetición** ocurre cuando el modelo encuentra demasiados registros repetidos y cree que ese patrón es **más importante de lo que realmente es**.

* **Origen:** fallos del sistema (doble registro de una transacción), reintentos de red, cargas duplicadas de archivos, uniones (*joins*) mal hechas.
* **Impacto en modelos no supervisados:** el clustering y la detección de anomalías son sensibles a la densidad; los duplicados inflan artificialmente la frecuencia de un comportamiento y distorsionan los grupos, los promedios y los umbrales de "normalidad".
* **Utilidad de la limpieza:** evita que el modelo se "obsesione" con un comportamiento que no representa la realidad.

A continuación se muestra primero la base sucia y luego la limpieza con `drop_duplicates()`.

In [ ]:
import pandas as pd

# 1. Base "sucia": compras repetidas por un fallo en el sistema
historial_compras = {
    'Cliente': ['Mateo', 'Valeria', 'Mateo', 'Diego', 'Mateo', 'Valeria'],
    'Articulo': ['Videojuego', 'Audífonos', 'Videojuego', 'Tenis', 'Videojuego', 'Audífonos'],
    'Precio_USD': [50, 35, 50, 80, 50, 35]
}
df = pd.DataFrame(historial_compras)

print("--- 🚨 BASE DE DATOS SUCIA (con sesgo por repetición) ---")
print(df)
print("\n¿Por qué está mal? Mateo solo compró 1 videojuego, pero aparece 3 veces.")
print("Si la IA lee esto, creerá falsamente que el videojuego se vende muchísimo más que los tenis.")

### Medir el sesgo antes de limpiar

In [ ]:
print("Filas duplicadas:", df.duplicated().sum(), "de", len(df))
print("\nVentas por artículo (sucia):")
print(df['Articulo'].value_counts())
print("\nIngreso total (sucia): $", df['Precio_USD'].sum())

### Deduplicación

In [ ]:
# .drop_duplicates() busca filas idénticas y conserva solo la primera
df_limpio = df.drop_duplicates()

print("--- ✅ BASE DE DATOS LIMPIA (deduplicada) ---")
print(df_limpio)
print("\n¡Listo! Ahora cada compra es única y la IA ya no tiene sesgo de repetición.")

### Comparación antes vs. después

In [ ]:
comparacion = pd.DataFrame({
    'Sucia': df['Articulo'].value_counts(),
    'Limpia': df_limpio['Articulo'].value_counts()
})
print(comparacion)
print("\nIngreso total (limpia): $", df_limpio['Precio_USD'].sum())
print("Ingreso inflado por duplicados: $", df['Precio_USD'].sum() - df_limpio['Precio_USD'].sum())

## Conclusión
* La base sucia mostraba al videojuego con **3 ventas** (la mayor frecuencia), cuando en realidad hubo **1**; el ingreso estaba inflado en **$135 USD** ($300 vs. $165).
* Tras `drop_duplicates()` los tres artículos tienen 1 compra real: desaparece la falsa "popularidad" del videojuego.
* **Precaución:** en datos reales, dos filas idénticas pueden ser compras legítimas distintas (p. ej. el mismo cliente compra dos veces el mismo artículo). Conviene incluir un identificador único (ID de transacción o *timestamp*) y deduplicar con `subset=[...]` antes de eliminar, y revisar los casos con `df[df.duplicated(keep=False)]`.